# 16장. 검색기 완성

| 항목 | 내용 |
|---|---|
| 예상 시간 | 4시간 |
| 선수 장 | 10장, 13장, 15장 |
| 이 장의 결과물 | `search(query, topk) -> list[(score, doc_id, snippet)]`, `SearchEngine` 클래스, CLI, 검색 품질 점검표 |
| 과제 연결 | 문서 검색 시스템(쿼리 벡터화, 코사인 유사도, 상위 5개), 실행 방법(안 A CLI, 안 B 함수) |

**학습 목표**
- 과제의 검색 함수 시그니처를 구현하고 결과를 과제 예시 형식으로 출력할 수 있다.
- 쿼리에 문서와 같은 전처리를 해야 하는 이유를 실패 사례로 설명할 수 있다.
- 동점과 잘못된 입력까지 고려해 상위 k개를 뽑을 수 있다.
- 쿼리 단어 주변을 단어 경계에서 잘라 스니펫을 만들 수 있다.
- `argparse`로 명령줄 검색 도구를 만들 수 있다.
- 여러 쿼리로 검색 품질을 점검하고, 실패한 쿼리를 BoW 한계와 연결해 설명할 수 있다.

In [1]:
# ── 7장에서 만든 데이터 로딩 코드 (src/data.py로 옮길 내용) ──
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

PROJECT = Path.cwd().parent if Path.cwd().name == "studys" else Path.cwd()
RANDOM_STATE = 42
EN_CATEGORIES = ["comp.graphics", "comp.windows.x", "rec.sport.baseball", "rec.sport.hockey", "sci.space"]
YNAT_URL = ("https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/"
            "klue_benchmark/ynat-v1.1/ynat-v1.1_train.json")
YNAT_PATH = PROJECT / "data" / "raw" / "ynat-v1.1_train.json"

_EMAIL = re.compile(r"\S+@\S+\.\S+")
_URL = re.compile(r"(https?://|www\.)\S+")
_PHONE = re.compile(r"\b\d{2,4}[-.\s]\d{3,4}[-.\s]\d{4}\b")


def mask_pii(text: str) -> str:
    text = _EMAIL.sub(" <email> ", text)
    text = _URL.sub(" <url> ", text)
    return _PHONE.sub(" <phone> ", text)


def finalize(df: pd.DataFrame) -> pd.DataFrame:
    """개인정보 가리기 → 빈 문서 제거 → 레이블 충돌 중복·중복 제거 → 인덱스 재정렬."""
    df = df.assign(text=df["text"].map(mask_pii).str.strip())
    df = df[df["text"] != ""]
    norm = df["text"].str.split().str.join(" ")
    conflict = df.groupby(norm)["label"].transform("nunique") > 1
    df, norm = df[~conflict], norm[~conflict]
    return df[~norm.duplicated(keep="first")].reset_index(drop=True)


def load_english() -> pd.DataFrame:
    bunch = fetch_20newsgroups(subset="all", categories=EN_CATEGORIES,
                               remove=("headers", "footers", "quotes"),
                               shuffle=True, random_state=RANDOM_STATE)
    labels = [bunch.target_names[t] for t in bunch.target]
    return finalize(pd.DataFrame({"text": bunch.data, "label": labels}))


def load_korean(per_class: int = 1000) -> pd.DataFrame:
    if not YNAT_PATH.exists():
        YNAT_PATH.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(YNAT_URL, YNAT_PATH)
    raw = pd.read_json(YNAT_PATH).rename(columns={"title": "text"})[["guid", "text", "label"]]
    sampled = (raw.groupby("label", group_keys=False)
                  .sample(n=per_class, random_state=RANDOM_STATE)
                  .sample(frac=1.0, random_state=RANDOM_STATE))
    return finalize(sampled.reset_index(drop=True))

# ── 8~10장에서 만든 전처리 파이프라인 (src/preprocess.py로 옮길 내용) ──
from konlpy.tag import Okt
from nltk.corpus import stopwords

PLACEHOLDER = re.compile(r"<(?:email|url|phone)>")
EN_TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")
KO_CLEAN = re.compile(r"[^0-9a-zA-Z가-힣一-鿿\s]")
HANJA_ONLY = re.compile(r"^[一-鿿]+$")
KO_KEEP_POS = {"Noun", "Verb", "Adjective", "Alpha"}
EN_STOP = frozenset(stopwords.words("english"))
DOMAIN_STOPWORDS_EN = frozenset({
    "would", "could", "might", "may", "also", "like", "get", "got", "go", "going", "know", "think",
    "see", "say", "make", "want", "look", "one", "well", "much", "many", "even", "way", "something",
    "anyone", "since", "still", "really", "right", "another", "cant",
})
STOPWORDS_KO = frozenset({
    "하다", "되다", "있다", "없다", "이다", "않다",
    "등", "것", "수", "중", "및", "때", "후", "내", "듯", "제", "개", "의", "로", "종합",
})
_okt = Okt()


def clean_en(text: str) -> str:
    text = PLACEHOLDER.sub(" ", text.lower()).replace("’", "'")
    return " ".join(EN_TOKEN.findall(text))


def clean_ko(text: str) -> str:
    return " ".join(KO_CLEAN.sub(" ", PLACEHOLDER.sub(" ", text)).split())


class EnglishPreprocessor:
    def __init__(self, use_domain_stopwords: bool = True, min_len: int = 2, max_len: int = 20):
        self.stopwords = EN_STOP | (DOMAIN_STOPWORDS_EN if use_domain_stopwords else frozenset())
        self.min_len, self.max_len = min_len, max_len

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for tok in clean_en(text).split():
            if tok in self.stopwords:
                continue
            tok = tok.removesuffix("'s").replace("'", "")
            if self.min_len <= len(tok) <= self.max_len and tok not in self.stopwords:
                tokens.append(tok)
        return tokens


class KoreanPreprocessor:
    def __init__(self, tokenizer: str = "okt", use_stopwords: bool = True):
        if tokenizer not in ("okt", "whitespace"):
            raise ValueError(f"tokenizer는 'okt' 또는 'whitespace': {tokenizer}")
        self.tokenizer = tokenizer
        self.stopwords = STOPWORDS_KO if use_stopwords else frozenset()

    def tokenize(self, text: str) -> list[str]:
        if self.tokenizer == "whitespace":
            return clean_ko(text).lower().split()
        return [w.lower() for w, p in _okt.pos(clean_ko(text), norm=True, stem=True)
                if p in KO_KEEP_POS or (p == "Foreign" and HANJA_ONLY.match(w))]

    def __call__(self, text: str) -> list[str]:
        return [t for t in self.tokenize(text) if t not in self.stopwords and not t.isdigit()]

# ── 로드 + 전처리를 한 번에 (src/corpus.py로 옮길 내용) ──
MIN_TOKENS = {"en": 5, "ko": 2}   # 전처리 후 토큰이 이보다 적은 문서는 뺀다 (짧은 답글, 주제어 없는 제목)
MIN_DF = {"en": 3, "ko": 2}       # 어휘에 넣을 최소 문서 빈도 (11장)


def load_corpus(lang: str, tokenizer: str = "okt") -> pd.DataFrame:
    """열: text, label, tokens. lang은 "en" 또는 "ko"."""
    df = load_english() if lang == "en" else load_korean()
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor(tokenizer)
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df

# ── 13장에서 만든 NumPy TF-IDF (src/tfidf.py로 옮길 내용) ──
class NumpyTfidf:
    """TF(raw 또는 log) × smooth IDF, 문서마다 L2 정규화. sklearn TfidfVectorizer와 같은 정의."""

    def __init__(self, tf: str = "raw", smooth_idf: bool = True, norm: str | None = "l2",
                 min_df: int = 1, max_df: float = 1.0):
        if tf not in ("raw", "log"):
            raise ValueError(f"tf는 'raw' 또는 'log': {tf}")
        if norm not in ("l2", None):
            raise ValueError(f"norm은 'l2' 또는 None: {norm}")
        self.tf, self.smooth_idf, self.norm = tf, smooth_idf, norm
        self.min_df, self.max_df = min_df, max_df

    def fit(self, docs: list[list[str]]) -> "NumpyTfidf":
        n = len(docs)
        df: dict[str, int] = {}
        for doc in docs:
            for t in set(doc):
                df[t] = df.get(t, 0) + 1
        max_count = self.max_df * n if isinstance(self.max_df, float) else self.max_df
        terms = sorted(t for t, c in df.items() if self.min_df <= c <= max_count)
        self.vocabulary_ = {t: i for i, t in enumerate(terms)}
        self.terms_ = np.array(terms, dtype=object)
        self.n_docs_ = n
        self.df_ = np.array([df[t] for t in terms], dtype=np.float64)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n) / (1.0 + self.df_)) + 1.0
        else:
            self.idf_ = np.log(n / self.df_) + 1.0
        return self

    def count_matrix(self, docs: list[list[str]]) -> np.ndarray:
        rows, cols = [], []
        for i, doc in enumerate(docs):
            for t in doc:
                j = self.vocabulary_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        counts = np.zeros((len(docs), len(self.vocabulary_)), dtype=np.float64)
        np.add.at(counts, (np.asarray(rows, dtype=np.int64), np.asarray(cols, dtype=np.int64)), 1.0)
        return counts

    def transform(self, docs: list[list[str]]) -> np.ndarray:
        x = self.count_matrix(docs)
        if self.tf == "log":
            mask = x > 0
            x[mask] = 1.0 + np.log(x[mask])
        x *= self.idf_                                   # (n, V) * (V,) 브로드캐스팅
        if self.norm == "l2":
            norms = np.linalg.norm(x, axis=1, keepdims=True)
            x /= np.where(norms > 0, norms, 1.0)          # 빈 문서는 0 벡터로 둔다
        return x

    def fit_transform(self, docs: list[list[str]]) -> np.ndarray:
        return self.fit(docs).transform(docs)

import time


def load_corpus_with_ids(lang: str) -> pd.DataFrame:
    """load_corpus와 같지만, 짧은 문서를 빼기 전에 문서 ID를 붙인다 (src/data.py와 같은 ID 규칙)."""
    df = load_english() if lang == "en" else load_korean()
    df.insert(0, "doc_id", [f"{lang}-{i:05d}" for i in range(len(df))])
    prep = EnglishPreprocessor() if lang == "en" else KoreanPreprocessor("okt")
    df["tokens"] = [prep(t) for t in df["text"]]
    df = df[df["tokens"].map(len) >= MIN_TOKENS[lang]].reset_index(drop=True)
    df.attrs["preprocessor"] = prep
    return df


t0 = time.perf_counter()
en = load_corpus_with_ids("en")
ko = load_corpus_with_ids("ko")
print(f"영어 {len(en):,}건, 한국어 {len(ko):,}건 ({time.perf_counter() - t0:.0f}초)")
en[["doc_id", "label", "text"]].head(3)

영어 4,646건, 한국어 7,000건 (11초)


,doc_id,label,text
0,en-00000,comp.windows.x,"According to the OSF/Motif Style Guide, one sh..."
1,en-00001,rec.sport.hockey,How was this trade bad? I seem to recall Ratel...
2,en-00002,rec.sport.hockey,He's not gone yet. The position opening is do...


**문서 ID**는 검색 결과에서 문서를 가리키는 이름입니다. 행 번호(0, 1, 2…)를 그대로 쓰면, 짧은 문서를 빼거나 데이터를 다시 정리할 때 번호가 바뀌어 예전 결과와 맞지 않게 됩니다. 그래서 **정리 직후, 토큰 수로 거르기 전에** `en-00000` 같은 ID를 붙여 고정합니다.

## 16.1 검색 함수 설계

과제가 요구하는 함수 모양은 이렇습니다.

```python
search(query: str) -> List[(score, doc_id, text_snippet)]
```

파이썬 타입 힌트로 정확히 쓰면 `def search(query: str, topk: int = 5) -> list[tuple[float, str, str]]`입니다(2장). 내부 흐름은 다음과 같습니다.

```
[미리 한 번]  문서 전체 → 전처리 → TF-IDF fit_transform → 정규화된 문서 행렬 X 저장
[쿼리마다]    쿼리 → 같은 전처리 → 같은 TF-IDF로 transform → q
              → 코사인 유사도 X @ q (15장) → 상위 k개 (16.3) → (점수, 문서 ID, 스니펫) (16.4)
```

**문서 쪽은 한 번만** 계산해 두고, 쿼리가 들어올 때마다 **쿼리 벡터 하나만** 새로 만듭니다. 쿼리로 TF-IDF를 다시 `fit`하면 어휘와 IDF가 바뀌어 문서 행렬과 비교할 수 없게 됩니다(2장 fit/transform).

## 16.2 쿼리에도 같은 전처리를

문서는 소문자 변환, 정제, 불용어 제거, `'s` 제거를 거쳐 색인되었습니다. 쿼리를 다르게 처리하면 어휘 사전의 단어와 **글자가 달라서** 하나도 맞지 않습니다.

In [2]:
vec_en = NumpyTfidf(min_df=MIN_DF["en"])
X_en = vec_en.fit_transform(en["tokens"].tolist())
prep_en = en.attrs["preprocessor"]

query = "Space Shuttle's LAUNCH!"
for name, toks in [("split()만", query.split()), ("문서와 같은 전처리", prep_en(query))]:
    known = [t for t in toks if t in vec_en.vocabulary_]
    q = vec_en.transform([toks])[0]
    print(f"{name:<14} 토큰 {toks} → 어휘에 있는 단어 {known}, 최고 점수 {(X_en @ q).max():.3f}")

split()만       토큰 ['Space', "Shuttle's", 'LAUNCH!'] → 어휘에 있는 단어 [], 최고 점수 0.000
문서와 같은 전처리     토큰 ['space', 'shuttle', 'launch'] → 어휘에 있는 단어 ['space', 'shuttle', 'launch'], 최고 점수 0.423


한국어는 더 극적입니다. 문서는 Okt 형태소로 색인했는데 쿼리를 공백으로 자르면, 조사가 붙은 어절이 어휘에 없습니다.

In [3]:
vec_ko = NumpyTfidf(min_df=MIN_DF["ko"])
X_ko = vec_ko.fit_transform(ko["tokens"].tolist())
prep_ko = ko.attrs["preprocessor"]

query = "스마트폰을 새로 출시했다"
for name, toks in [("공백 토큰화", KoreanPreprocessor("whitespace")(query)), ("문서와 같은 Okt", prep_ko(query))]:
    known = [t for t in toks if t in vec_ko.vocabulary_]
    q = vec_ko.transform([toks])[0]
    print(f"{name:<12} 토큰 {toks}\n{'':<12} → 어휘에 있는 단어 {known}, 최고 점수 {(X_ko @ q).max():.3f}")

공백 토큰화       토큰 ['스마트폰을', '새로', '출시했다']
             → 어휘에 있는 단어 [], 최고 점수 0.000
문서와 같은 Okt   토큰 ['스마트폰', '새롭다', '출시']
             → 어휘에 있는 단어 ['스마트폰', '새롭다', '출시'], 최고 점수 0.482


그래서 검색 엔진은 **문서를 색인할 때 쓴 전처리 객체를 그대로 보관**해 두었다가 쿼리에도 씁니다. 10장에서 전처리를 `__call__` 하나로 묶은 이유가 여기에 있습니다.

## 16.3 상위 k개 뽑기

4장에서 배운 `argsort`를 씁니다. 세 가지를 신경 씁니다.
1. **동점**: `kind="stable"`로 문서 순서를 유지해 실행할 때마다 같은 결과가 나오게 합니다.
2. **잘못된 k**: `k < 1`이면 에러를 내고, 문서 수보다 크면 문서 수만큼만 돌려줍니다.
3. **점수 0인 문서**: 쿼리 단어가 하나도 없는 문서는 결과에서 뺍니다. "관련 없음"을 5위 안에 채워 넣으면 사용자를 속이게 됩니다.

문서가 아주 많을 때는 `argpartition`으로 후보 k개만 고른 뒤 정렬하면 빠릅니다(4장 4.8절). 여기서는 두 방법을 모두 구현하고 결과가 같은지 확인합니다.

In [4]:
def top_k(scores: np.ndarray, k: int) -> np.ndarray:
    """점수 상위 k개 인덱스 (내림차순, 동점은 앞 문서 먼저)."""
    if k < 1:
        raise ValueError(f"k는 1 이상이어야 합니다: {k}")
    k = min(k, len(scores))
    cand = np.argpartition(-scores, k - 1)[:k]             # 순서 없는 상위 k개 후보 (O(n))
    return cand[np.argsort(-scores[cand], kind="stable")]  # 후보 k개만 정렬


scores = np.array([0.1, 0.9, 0.0, 0.9, 0.5])
print("top_k(scores, 3) :", top_k(scores, 3))
print("top_k(scores, 99):", top_k(scores, 99))
try:
    top_k(scores, 0)
except ValueError as e:
    print("top_k(scores, 0) → ValueError:", e)

rng = np.random.default_rng(0)
big = rng.random(len(en))
print("argsort 결과와 같은가:", np.array_equal(top_k(big, 5), np.argsort(-big, kind="stable")[:5]))

top_k(scores, 3) : [1 3 4]
top_k(scores, 99): [1 3 4 0 2]
top_k(scores, 0) → ValueError: k는 1 이상이어야 합니다: 0
argsort 결과와 같은가: True


> 점수가 같은 문서가 k번째 경계에 걸쳐 있으면 `argpartition`이 어느 문서를 후보에 넣을지는 보장되지 않습니다. 실제 TF-IDF 점수에서 정확히 같은 값이 경계에 걸리는 일은 드물지만, 완전히 같은 결과가 꼭 필요하면 `np.argsort(-scores, kind="stable")[:k]` 하나로 충분합니다.

## 16.4 스니펫: 쿼리 단어 주변을 보여 주기

검색 결과에는 문서 전체가 아니라 **짧은 발췌(스니펫)**를 보여 줍니다. 문서 맨 앞을 자르면 쿼리와 관련 없는 인사말만 보일 수 있으므로, **쿼리 단어가 처음 나오는 곳 주변**을 자릅니다.

단순히 `text.find("art")`로 찾으면 `start` 안의 `art`에 걸립니다. 그래서 정규식 `(?<!\w)단어`로 **단어가 시작하는 위치**에서만 찾습니다. `(?<!\w)`는 "바로 앞 글자가 단어 글자가 아니다"라는 뜻입니다. 끝은 막지 않아서 쿼리 `launch`가 본문의 `launched`에서도 찾아집니다.

In [5]:
SNIPPET_LEN = 90


def make_snippet(text: str, query_tokens: list[str], width: int = SNIPPET_LEN) -> str:
    """쿼리 단어가 처음 나오는 곳을 중심으로 자른 한 줄 요약. 못 찾으면 앞부분."""
    flat = " ".join(text.split())                              # 줄바꿈·연속 공백을 한 칸으로
    lower = flat.lower()
    hits = [m.start() for t in query_tokens if (m := re.search(rf"(?<!\w){re.escape(t)}", lower))]
    start = max(0, min(hits) - width // 3) if hits else 0      # 찾은 위치보다 조금 앞에서 시작
    piece = flat[start:start + width]
    return ("…" if start > 0 else "") + piece + ("…" if start + width < len(flat) else "")


demo = "We should start the art exhibit after the shuttle launched safely last week."
print("text.find('art')        :", demo.find("art"), "→", demo[demo.find("art") - 2:demo.find("art") + 8])
print("make_snippet(['art'])   :", make_snippet(demo, ["art"], width=30))
print("make_snippet(['launch']):", make_snippet(demo, ["launch"], width=30))

text.find('art')        : 12 → start the 
make_snippet(['art'])   : …start the art exhibit after th…
make_snippet(['launch']): …e shuttle launched safely last…


- `find`는 `start`의 일부인 `art`를 찾았지만, `make_snippet`은 단어 `art`를 찾았습니다.
- 앞뒤가 잘렸으면 `…`를 붙여서 발췌라는 것을 표시합니다.
- 한국어는 형태소 토큰(`삼성`, `실적`)이 원문 어절(`삼성전자가`, `실적을`)의 **앞부분**이므로, 같은 방식으로 찾을 수 있습니다.

## 16.5 SearchEngine 클래스

지금까지의 조각을 하나로 묶습니다. 생성할 때 문서 행렬을 한 번 만들고, `search`는 쿼리만 처리합니다.

In [6]:
def cosine_similarity(doc_matrix: np.ndarray, query_vec: np.ndarray) -> np.ndarray:   # 15장
    dots = doc_matrix @ query_vec
    denom = np.linalg.norm(doc_matrix, axis=1) * np.linalg.norm(query_vec)
    return np.divide(dots, denom, out=np.zeros_like(dots), where=denom > 0)


class SearchEngine:
    def __init__(self, corpus: pd.DataFrame, preprocessor, vectorizer: NumpyTfidf):
        self.corpus = corpus
        self.prep = preprocessor                       # 문서에 쓴 전처리를 쿼리에도 그대로 쓴다
        self.vec = vectorizer
        self.matrix = vectorizer.fit_transform(corpus["tokens"].tolist())   # 한 번만 계산

    def query_vector(self, query: str) -> tuple[np.ndarray, list[str]]:
        tokens = self.prep(query)
        return self.vec.transform([tokens])[0], tokens

    def search(self, query: str, topk: int = 5) -> list[tuple[float, str, str]]:
        if topk < 1:
            raise ValueError(f"topk는 1 이상이어야 합니다: {topk}")
        q, tokens = self.query_vector(query)
        if not q.any():                                # 불용어·OOV만 있는 쿼리
            return []
        scores = cosine_similarity(self.matrix, q)
        results = []
        for i in top_k(scores, topk):
            if scores[i] <= 0:                         # 쿼리 단어가 하나도 없는 문서는 빼기
                break
            row = self.corpus.iloc[i]
            results.append((float(scores[i]), row["doc_id"], make_snippet(row["text"], tokens)))
        return results


def format_results(query: str, results, labels: dict[str, str] | None = None) -> str:
    """과제 결과 예시 형식으로 출력한다."""
    lines = ["=== 검색 결과 ===", f'Query: "{query}"', "", f"Top {len(results)} 결과:"]
    if not results:
        lines.append("  (일치하는 문서 없음: 검색어의 단어가 모두 불용어이거나 어휘 사전에 없습니다)")
    for rank, (score, doc_id, snippet) in enumerate(results, 1):
        tag = f" [{labels[doc_id]}]" if labels else ""
        lines.append(f'{rank}. [{score:.3f}] {doc_id}{tag} "{snippet}"')
    return "\n".join(lines)


t0 = time.perf_counter()
engine_en = SearchEngine(en, prep_en, NumpyTfidf(min_df=MIN_DF["en"]))
engine_ko = SearchEngine(ko, prep_ko, NumpyTfidf(min_df=MIN_DF["ko"]))
print(f"색인 생성: {time.perf_counter() - t0:.1f}초")
labels_en = dict(zip(en["doc_id"], en["label"]))
labels_ko = dict(zip(ko["doc_id"], ko["label"]))

t0 = time.perf_counter()
res = engine_en.search("space shuttle launch", topk=5)
print(f"쿼리 한 번: {(time.perf_counter() - t0) * 1000:.0f}ms\n")
print(format_results("space shuttle launch", res, labels_en))

색인 생성: 0.4초
쿼리 한 번: 78ms

=== 검색 결과 ===
Query: "space shuttle launch"

Top 5 결과:
1. [0.423] en-03186 [sci.space] "Well, you better not get the shuttle as your launch vehicle. and most ELV's have too far o…"
2. [0.418] en-01889 [sci.space] "COMMERCIAL SPACE NEWS/SPACE TECHNOLOGY INVESTOR NUMBER 22 This is number twenty-two in an …"
3. [0.406] en-04447 [sci.space] "…st exercise is analogous to a shuttle landing/launch in terms of stresses/misalignments/et…"
4. [0.406] en-01329 [sci.space] "…d on the technical aspects of space, but I couldn't find the answer on the FAQs ! I'm curr…"
5. [0.399] en-00039 [sci.space] "…essay, I am writing about the space shuttle and a need for a better propulsion system. Thr…"


In [7]:
print(format_results("삼성전자 반도체 실적", engine_ko.search("삼성전자 반도체 실적"), labels_ko))

=== 검색 결과 ===
Query: "삼성전자 반도체 실적"

Top 5 결과:
1. [0.437] ko-06717 [경제] "키움증권 삼성전자 3분기 사상최대 실적…영업익 17조8천억"
2. [0.417] ko-05504 [경제] "유진투자 삼성전자 3분기 실적 개선 전망…목표주가↑"
3. [0.389] ko-01858 [경제] "영상 삼성전자 60조 흑자 시대 개막 미뤘다…4분기는 실적 충격"
4. [0.384] ko-05845 [경제] "삼성증권 삼성전자 2분기 영업익 6조9천억원 추정"
5. [0.376] ko-05553 [경제] "대신증권 삼성전자 2분기 실적 예상치 웃돌아…목표가↑"


색인은 처음 한 번만 만들고, 이후 쿼리는 밀리초 단위로 처리됩니다.

### 과제의 "안 B" 함수 형태
과제는 `search(query)`처럼 **쿼리 하나만 받는 함수**도 허용합니다. 엔진을 매번 만들면 느리므로, 처음 호출할 때 한 번 만들어 두고 재사용합니다(`functools.lru_cache`).

In [8]:
from functools import lru_cache


@lru_cache(maxsize=2)
def get_engine(lang: str = "en") -> SearchEngine:
    corpus = en if lang == "en" else ko            # 실제 src/search.py에서는 여기서 load_corpus를 호출한다
    return SearchEngine(corpus, corpus.attrs["preprocessor"], NumpyTfidf(min_df=MIN_DF[lang]))


def search(query: str, topk: int = 5, lang: str = "en") -> list[tuple[float, str, str]]:
    """검색 함수 (안 B): search(query) -> List[(score, doc_id, text_snippet)]"""
    return get_engine(lang).search(query, topk)


t = time.perf_counter()
search("hubble telescope mirror")
first = time.perf_counter() - t
t = time.perf_counter()
out = search("hubble telescope mirror", topk=3)
print(f"첫 호출 {first:.2f}초 (색인 생성 포함), 두 번째 {(time.perf_counter() - t) * 1000:.0f}ms")
for score, doc_id, snippet in out:
    print(f"({score:.3f}, {doc_id!r}, {snippet[:60]!r}...)")

첫 호출 0.46초 (색인 생성 포함), 두 번째 78ms
(0.392, 'en-01360', '…s, but have you looked ad the Hubble results, the defects o'...)
(0.344, 'en-00411', '…all >1g forces applied to the mirror and its mounting (and '...)
(0.265, 'en-01186', '…ion problems with the primary mirror was to drive the compu'...)


## 16.6 명령줄 도구(CLI) 만들기

과제의 "안 A"는 `python main.py --query "..." --topk 5` 같은 명령으로 실행하는 방식입니다. 파이썬 표준 라이브러리 `argparse`가 명령줄 인자를 읽어 줍니다.

| 코드 | 의미 |
|---|---|
| `ap.add_argument("--query")` | `--query 값` 형태의 인자 (없으면 None) |
| `ap.add_argument("--topk", type=int, default=5)` | 정수로 바꿔 읽고, 없으면 5 |
| `ap.add_argument("--lang", choices=["en", "ko"])` | 정해진 값만 허용 |
| `ap.add_argument("--explain", action="store_true")` | 붙이면 True인 스위치 |
| `ap.error("메시지")` | 사용법과 함께 에러를 출력하고 종료 |

CLI의 구조만 연습하려고, 미니 코퍼스를 검색하는 작은 스크립트를 임시 폴더에 만들어 실행해 봅니다.

In [9]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

cli_dir = Path(tempfile.mkdtemp())
(cli_dir / "mini_search.py").write_text("""# 미니 코퍼스 검색 CLI (argparse 연습용)
import argparse

DOCS = ["The rocket launched into space!", "NASA launched a new space telescope.",
        "The pitcher threw a fast ball.", "Our team won the baseball game.",
        "Astronauts watched the baseball game in space."]


def main() -> None:
    ap = argparse.ArgumentParser(description="미니 코퍼스 검색")
    ap.add_argument("--query", help="검색어")
    ap.add_argument("--topk", type=int, default=5, help="결과 개수 (기본 5)")
    args = ap.parse_args()
    if not args.query:
        ap.error("--query 가 필요합니다.")
    if args.topk < 1:
        ap.error(f"--topk 는 1 이상이어야 합니다: {args.topk}")
    words = set(args.query.lower().split())
    scored = [(len(words & set(d.lower().strip(".!").split())), i, d) for i, d in enumerate(DOCS)]
    scored = sorted([s for s in scored if s[0] > 0], key=lambda s: (-s[0], s[1]))[: args.topk]
    print(f'Query: "{args.query}"')
    for rank, (score, i, d) in enumerate(scored, 1):
        print(f"{rank}. [{score}] doc {i}: {d}")


if __name__ == "__main__":
    main()
""", encoding="utf-8")


def run_cli(*args):
    env = {**os.environ, "PYTHON_COLORS": "0", "NO_COLOR": "1"}   # 터미널 색상 코드 끄기
    r = subprocess.run([sys.executable, "mini_search.py", *args], cwd=cli_dir,
                       capture_output=True, text=True, env=env)
    print(f"$ python mini_search.py {' '.join(repr(a) if ' ' in a else a for a in args)}")
    print((r.stdout + r.stderr).rstrip(), "\n")


run_cli("--query", "space game", "--topk", "2")
run_cli("--topk", "3")
run_cli("--query", "space", "--topk", "0")
run_cli("--help")

$ python mini_search.py --query 'space game' --topk 2
Query: "space game"
1. [2] doc 4: Astronauts watched the baseball game in space.
2. [1] doc 0: The rocket launched into space! 

$ python mini_search.py --topk 3
usage: mini_search.py [-h] [--query QUERY] [--topk TOPK]
mini_search.py: error: --query 가 필요합니다. 

$ python mini_search.py --query space --topk 0
usage: mini_search.py [-h] [--query QUERY] [--topk TOPK]
mini_search.py: error: --topk 는 1 이상이어야 합니다: 0 

$ python mini_search.py --help
usage: mini_search.py [-h] [--query QUERY] [--topk TOPK]

미니 코퍼스 검색

options:
  -h, --help     show this help message and exit
  --query QUERY  검색어
  --topk TOPK    결과 개수 (기본 5) 



- `--query`가 없거나 `--topk`가 0이면 `ap.error`가 사용법과 에러 메시지를 출력하고 종료합니다(종료 코드 2).
- `--help`는 `argparse`가 자동으로 만들어 줍니다.

### 프로젝트의 `main.py`
실제 과제에서는 `main.py`가 `--mode`로 기능을 고르고, 검색 모드에서 위의 `SearchEngine`과 `format_results`를 부릅니다. 프로젝트 폴더에서 이렇게 실행합니다.

```bash
uv run python main.py --query "space shuttle launch" --topk 5
uv run python main.py --lang ko --query "삼성전자 반도체 실적" --explain   # 1위 문서의 계산 과정까지
uv run python main.py --lang ko --tokenizer whitespace --query "삼성전자 실적"
```

README의 "실행 방법"에는 이 명령과 **실제 출력 예시**를 함께 붙입니다. 과제는 안 A(CLI)와 안 B(함수) 중 하나 이상을 요구하지만, 둘 다 적어 두면 평가자가 고르기 쉽습니다.

## 16.7 검색 품질 점검

검색이 "잘 된다"를 말로만 하지 않고 숫자로 확인합니다. 정답 문서 목록이 없으므로, 쿼리마다 **기대하는 카테고리**를 정해 두고 상위 5개 중 그 카테고리 문서의 비율(**precision@5**)을 셉니다. 간단하지만 쿼리마다 비교하기에 충분합니다.

In [10]:
def precision_at_k(engine, labels, query, expected, k=5) -> tuple[float, int]:
    res = engine.search(query, k)
    hits = sum(labels[doc_id] == expected for _, doc_id, _ in res)
    return (hits / len(res) if res else 0.0), len(res)


checks_en = [
    ("space shuttle launch", "sci.space"),
    ("hubble telescope mirror", "sci.space"),
    ("pitcher inning home run", "rec.sport.baseball"),
    ("goalie penalty shot", "rec.sport.hockey"),
    ("3d rendering software", "comp.graphics"),
    ("window server font", "comp.windows.x"),
]
checks_ko = [
    ("삼성전자 반도체 실적", "경제"),
    ("북한 미사일 발사", "정치"),
    ("손흥민 골", "스포츠"),
    ("대통령 정상회담", "정치"),
    ("스마트폰 출시", "IT과학"),
    ("미국 대선 트럼프", "세계"),
]
rows = []
for engine, labels, checks in [(engine_en, labels_en, checks_en), (engine_ko, labels_ko, checks_ko)]:
    for q, expected in checks:
        p, n = precision_at_k(engine, labels, q, expected)
        rows.append({"쿼리": q, "전처리 후 토큰": engine.prep(q), "기대 카테고리": expected,
                     "결과 수": n, "precision@5": p})
quality = pd.DataFrame(rows)
quality

,쿼리,전처리 후 토큰,기대 카테고리,결과 수,precision@5
0,space shuttle launch,"[space, shuttle, launch]",sci.space,5,1.0
1,hubble telescope mirror,"[hubble, telescope, mirror]",sci.space,5,0.8
2,pitcher inning home run,"[pitcher, inning, home, run]",rec.sport.baseball,5,1.0
3,goalie penalty shot,"[goalie, penalty, shot]",rec.sport.hockey,5,1.0
4,3d rendering software,"[rendering, software]",comp.graphics,5,0.8
5,window server font,"[window, server, font]",comp.windows.x,5,1.0
6,삼성전자 반도체 실적,"[삼성, 전자, 반도체, 실적]",경제,5,1.0
7,북한 미사일 발사,"[북한, 미사일, 발사]",정치,5,1.0
8,손흥민 골,"[손흥민, 골]",스포츠,5,1.0
9,대통령 정상회담,"[대통령, 정상회담]",정치,5,0.8


In [11]:
print(f"평균 precision@5 = {quality['precision@5'].mean():.2f}")
for q in ["goalie penalty shot", "스마트폰 출시"]:
    engine, labels = (engine_en, labels_en) if q.isascii() else (engine_ko, labels_ko)
    print()
    print(format_results(q, engine.search(q), labels))

평균 precision@5 = 0.92

=== 검색 결과 ===
Query: "goalie penalty shot"

Top 5 결과:
1. [0.259] en-01436 [rec.sport.hockey] "… the ref, by player or coach. Penalty shot is awarded. Two and you are thrown out of the g…"
2. [0.237] en-03988 [rec.sport.hockey] "…s made. The average number of shots taken by each team in a hockey game (and this is a tot…"
3. [0.220] en-04307 [rec.sport.hockey] "Yes, he does. BUT, the goalie sure as hell doesn't want him there! When I played roller ho…"
4. [0.208] en-04305 [rec.sport.hockey] "…personal favorite move (I'm a goalie too) is to give the offeding player a good whack on t…"
5. [0.201] en-01421 [rec.sport.hockey] "… games Finns have played, the goalie has been chosen the best player of the opposing team.…"

=== 검색 결과 ===
Query: "스마트폰 출시"

Top 5 결과:
1. [0.506] ko-02742 [IT과학] "20만원대 스마트폰 LG X401 7일 출시…전면 광각 카메라"
2. [0.480] ko-01887 [IT과학] "SKT 전용 스마트폰 갤럭시 와이드 4 출시…28만원대"
3. [0.448] ko-03225 [IT과학] "스마트폰 버전 슈퍼마리오 한국에 지각 출시"
4. [0.408] ko-06645 [IT과학] "삼성·LG 

대부분의 쿼리에서 상위 결과가 기대한 카테고리에 모입니다. 단어가 주제를 분명하게 가리키면 BoW 검색만으로도 꽤 잘 동작합니다. precision@5가 낮은 쿼리가 있다면 결과를 직접 읽고, 다른 카테고리 문서가 **어떤 단어 때문에** 올라왔는지 확인하세요(16.9절 방법). 대개 여러 주제에서 함께 쓰이는 단어가 원인입니다.

## 16.8 실패하는 쿼리 모으기

잘 되는 쿼리보다 **실패하는 쿼리**가 리포트에 더 중요합니다. BoW 한계(21장)의 구체적 사례가 되기 때문입니다.

### ① 동의어와 표기 차이

In [12]:
def top_ids(engine, q, k=10):
    return [doc_id for _, doc_id, _ in engine.search(q, k)]


pairs = [(engine_ko, labels_ko, "북한", "北"), (engine_ko, labels_ko, "미국", "美"),
         (engine_en, labels_en, "nasa shuttle launch", "space agency orbiter liftoff")]
for engine, labels, a, b in pairs:
    ra, rb = top_ids(engine, a), top_ids(engine, b)
    print(f"{a!r} vs {b!r}: 토큰 {engine.prep(a)} / {engine.prep(b)}")
    print(f"  상위 10개 중 겹치는 문서: {len(set(ra) & set(rb))}개")

'북한' vs '北': 토큰 ['북한'] / ['北']
  상위 10개 중 겹치는 문서: 0개
'미국' vs '美': 토큰 ['미국'] / ['美']
  상위 10개 중 겹치는 문서: 0개


'nasa shuttle launch' vs 'space agency orbiter liftoff': 토큰 ['nasa', 'shuttle', 'launch'] / ['space', 'agency', 'orbiter', 'liftoff']
  상위 10개 중 겹치는 문서: 0개


`북한`과 `北`은 뜻이 같지만 BoW에서는 **다른 차원**이라, 두 쿼리의 결과가 거의 겹치지 않습니다. 영어도 같은 사건을 다른 단어로 쓰면(`nasa`→`space agency`, `shuttle`→`orbiter`, `launch`→`liftoff`) 다른 문서가 나옵니다. 10장에서 동의어 사전을 쓰지 않기로 한 결과이고, BoW의 근본 한계입니다.

### ② 같은 글자, 다른 의미

In [13]:
print(format_results("북한", engine_ko.search("북한", 5), labels_ko))
print("\n'북한산' 제목의 토큰:", prep_ko("설산으로 변한 북한산은 한 폭의 수채화"))

=== 검색 결과 ===
Query: "북한"

Top 5 결과:
1. [0.493] ko-01728 [정치] "북한 목선 인양준비 작업"
2. [0.443] ko-06326 [정치] "북한 김정일 생일 경축 불꽃놀이"
3. [0.409] ko-05125 [생활문화] "설산으로 변한 북한산은 한 폭의 수채화"
4. [0.401] ko-02069 [정치] "북한에 허가받은 상설시장 450개…평남 최다"
5. [0.391] ko-03940 [정치] "북한 김정은 낮 12시 30분 신년사 발표"

'북한산' 제목의 토큰: ['설산', '변하다', '북한', '산', '폭', '수채화']


`북한산`(서울의 산)이 형태소 분석에서 `북한` + `산`으로 쪼개져, 산 이야기가 "북한" 검색 결과에 섞입니다. 단어의 글자만 보고 문맥을 보지 않기 때문입니다.

### ③ 처음 보는 단어 (OOV)

In [14]:
for q in ["코로나 확진자", "메타버스 플랫폼"]:
    toks = prep_ko(q)
    print(f"{q!r} → 토큰 {toks}, 어휘에 있는 단어 {[t for t in toks if t in engine_ko.vec.vocabulary_]}")
    print(format_results(q, engine_ko.search(q, 3), labels_ko), "\n")

'코로나 확진자' → 토큰 ['코로나', '확', '진자'], 어휘에 있는 단어 ['확']
=== 검색 결과 ===
Query: "코로나 확진자"

Top 3 결과:
1. [0.486] ko-00228 [경제] "한국 조선 7년 만에 연간 수주량 세계 1위 확실시"
2. [0.406] ko-05692 [세계] "포르투갈 총선 종료…집권 사회당 제1당 확실시"
3. [0.385] ko-02682 [IT과학] "네이버 뉴스 사람 손길 확 줄였다…뉴스·섹션홈 AI 편집" 

'메타버스 플랫폼' → 토큰 ['메타', '버스', '플랫폼'], 어휘에 있는 단어 ['버스', '플랫폼']
=== 검색 결과 ===
Query: "메타버스 플랫폼"

Top 3 결과:
1. [0.405] ko-00452 [생활문화] "국립상당산성휴양림 버스로 편리하게 이용하세요"
2. [0.346] ko-06843 [IT과학] "5GX MEC 플랫폼이란"
3. [0.308] ko-02279 [경제] "KT 2021년까지 5대 플랫폼 매출 비중 30%로 확대" 



데이터(2016~2020년 초 기사 제목)에 `코로나`가 없어서 이 단어는 무시됩니다. 남은 것이 형태소 분석기가 잘못 자른 조각이라면, 그 조각과 우연히 겹치는 **전혀 관계없는 기사**가 결과로 나옵니다. `메타버스`도 `메타`+`버스`로 쪼개져 `버스`만 남고, 휴양림 버스 안내 기사가 1위가 됩니다. 사용자는 왜 이런 결과가 나왔는지 알기 어렵습니다. 색인에 없는 단어는 찾을 수 없고, 새 단어가 생기면 색인(어휘와 IDF)을 다시 만들어야 합니다.

### ④ 불용어만 있는 쿼리

In [15]:
for engine, labels, q in [(engine_en, labels_en, "would you like to know"), (engine_ko, labels_ko, "하는 것이 있다")]:
    print(f"토큰: {engine.prep(q)}")
    print(format_results(q, engine.search(q), labels), "\n")

토큰: []
=== 검색 결과 ===
Query: "would you like to know"

Top 0 결과:
  (일치하는 문서 없음: 검색어의 단어가 모두 불용어이거나 어휘 사전에 없습니다) 

토큰: []
=== 검색 결과 ===
Query: "하는 것이 있다"

Top 0 결과:
  (일치하는 문서 없음: 검색어의 단어가 모두 불용어이거나 어휘 사전에 없습니다) 



모든 단어가 불용어라 쿼리 벡터가 0이 되고, 빈 결과와 안내 메시지가 나옵니다. 15.6절의 0 벡터 처리가 실제로 쓰이는 경우입니다.

| 실패 유형 | 예 | 원인 | 21장 BoW 한계 |
|---|---|---|---|
| 동의어·표기 차이 | `북한` vs `北`, `launch` vs `liftoff` | 다른 단어는 다른 차원 | 동의어 |
| 같은 글자 다른 의미 | `북한` → `북한산` | 글자만 보고 문맥을 보지 않음 | 동음이의어 |
| OOV | `코로나` | 색인 시점에 없던 단어 | 어휘 고정 |
| 불용어만 | `would you like to know` | 남는 단어 없음 | (설계상 빈 결과) |

## 16.9 왜 이 문서가 1위인가: 단어별 기여도

정규화된 벡터에서 코사인 유사도는 $\sum_t q_t d_t$이므로, **단어 $t$의 기여도 $q_t d_t$**로 나눠서 설명할 수 있습니다. 결과를 사람이 이해할 수 있게 만드는 가장 직접적인 방법입니다.

In [16]:
def explain(engine, query: str, rank: int = 1) -> None:
    q, tokens = engine.query_vector(query)
    scores = cosine_similarity(engine.matrix, q)
    i = top_k(scores, rank)[rank - 1]
    d = engine.matrix[i]
    terms, idf = engine.vec.terms_, engine.vec.idf_
    unknown = [t for t in tokens if t not in engine.vec.vocabulary_]
    row = engine.corpus.iloc[i]
    print("=== 코사인 유사도 계산 예시 ===")
    print(f'Query: "{query}"')
    print(f"전처리 후 토큰: {tokens}" + (f"  (어휘에 없어 버린 단어: {unknown})" if unknown else ""))
    print(f"대상 문서: {rank}위 {row['doc_id']} [{row['label']}] \"{make_snippet(row['text'], tokens, 60)}\"")
    print(f"벡터 차원: {len(q):,} (쿼리에서 0이 아닌 차원 {np.count_nonzero(q)}개, 문서에서 {np.count_nonzero(d)}개)\n")
    print(f"{'단어':<12}{'IDF':>8}{'q 가중치':>10}{'d 가중치':>10}{'q×d':>10}")
    for j in np.flatnonzero(q):
        print(f"{terms[j]:<12}{idf[j]:>8.3f}{q[j]:>10.4f}{d[j]:>10.4f}{q[j] * d[j]:>10.4f}")
    print(f"\n코사인 유사도 = Σ q×d = {q @ d:.4f}  (정규화된 벡터라 분모는 1 × 1)")


explain(engine_ko, "삼성전자 반도체 실적")

=== 코사인 유사도 계산 예시 ===
Query: "삼성전자 반도체 실적"
전처리 후 토큰: ['삼성', '전자', '반도체', '실적']
대상 문서: 1위 ko-06717 [경제] "키움증권 삼성전자 3분기 사상최대 실적…영업익 17조8천억"
벡터 차원: 6,332 (쿼리에서 0이 아닌 차원 4개, 문서에서 9개)

단어               IDF     q 가중치     d 가중치       q×d
반도체            6.909    0.5879    0.0000    0.0000
삼성             4.678    0.3980    0.2656    0.1057
실적             6.243    0.5312    0.3545    0.1883
전자             5.435    0.4624    0.3086    0.1427

코사인 유사도 = Σ q×d = 0.4367  (정규화된 벡터라 분모는 1 × 1)


표를 읽는 법:
- **IDF**가 클수록 드문 단어입니다. 쿼리 안에서도 드문 단어가 더 큰 가중치(`q 가중치`)를 받습니다.
- `d 가중치`가 0인 단어는 1위 문서에 없는 단어입니다. 쿼리 단어를 모두 가진 문서가 아니어도, 가진 단어의 비중이 크면 1위가 될 수 있습니다.
- 마지막 열을 모두 더하면 점수가 됩니다. "이 문서는 `실적`과 `전자` 덕분에 1위"처럼 말할 수 있습니다.

In [17]:
explain(engine_en, "space shuttle launch")

=== 코사인 유사도 계산 예시 ===
Query: "space shuttle launch"
전처리 후 토큰: ['space', 'shuttle', 'launch']
대상 문서: 1위 en-03186 [sci.space] "… better not get the shuttle as your launch vehicle. and most…"
벡터 차원: 13,387 (쿼리에서 0이 아닌 차원 3개, 문서에서 21개)

단어               IDF     q 가중치     d 가중치       q×d
launch         4.656    0.6284    0.3380    0.2124
shuttle        4.640    0.6262    0.3368    0.2109
space          3.421    0.4616    0.0000    0.0000

코사인 유사도 = Σ q×d = 0.4233  (정규화된 벡터라 분모는 1 × 1)


영어 1위 문서에는 `space`가 없습니다(`d 가중치` 0). 그래도 짧은 글에서 `shuttle`과 `launch`의 비중이 커서, `space`까지 가진 긴 글보다 점수가 높습니다. 쿼리 단어를 **모두** 가진 문서가 1위가 되는 것은 아니라는 점을 기억하세요.

## 📝 리포트용 정리: 검색 시스템과 실행 방법

**구현 요약**
1. 문서 전체를 전처리하고 NumPy TF-IDF로 `fit_transform` → L2 정규화된 문서 행렬을 한 번 만든다.
2. 쿼리를 **같은 전처리 객체**로 토큰화하고, 문서로 학습한 어휘·IDF로 `transform`한다. 어휘에 없는 단어는 버린다.
3. 코사인 유사도를 `X @ q` 한 번으로 계산한다. 0 벡터 쿼리는 빈 결과를 돌려준다.
4. `argpartition` + stable `argsort`로 상위 k개를 고르고, 점수 0인 문서는 뺀다.
5. `(점수, 문서 ID, 스니펫)`을 돌려준다. 스니펫은 쿼리 단어가 처음 나오는 곳 주변을 자른다.

**실행 방법**

```bash
# 안 A: CLI (프로젝트 폴더에서)
uv run python main.py --query "space shuttle launch" --topk 5
uv run python main.py --lang ko --query "삼성전자 반도체 실적" --explain
```

```python
# 안 B: 함수
from src.search import search
search("space shuttle launch", topk=5)   # -> [(score, doc_id, snippet), ...]
```

**결과와 한계**: 16.7절 precision@5 표, 16.8절 실패 쿼리 표, 16.9절 기여도 분해를 실제 출력과 함께 붙인다.

## ✍️ 실습 1: 쿼리 단어 강조 스니펫

`make_snippet`을 바탕으로, 스니펫 안에서 쿼리 단어로 **시작하는** 부분을 `[` `]`로 감싸는 `highlight_snippet(text, query_tokens, width=90)`을 작성하세요. 대소문자와 상관없이 찾고, 원문의 대소문자는 그대로 둡니다.

예: `highlight_snippet("The Shuttle launched today", ["shuttle", "launch"])` → `"The [Shuttle] [launch]ed today"`

In [18]:
# 여기에 작성하세요

**정답**

In [19]:
def highlight_snippet(text: str, query_tokens: list[str], width: int = SNIPPET_LEN) -> str:
    snippet = make_snippet(text, query_tokens, width)
    for t in sorted(set(query_tokens), key=len, reverse=True):          # 긴 단어부터 감싼다
        snippet = re.sub(rf"(?<![\w\[])({re.escape(t)})", r"[\1]", snippet, flags=re.IGNORECASE)
    return snippet


print(highlight_snippet("The Shuttle launched today", ["shuttle", "launch"]))
for score, doc_id, _ in engine_en.search("hubble telescope mirror", 2):
    text = en.loc[en["doc_id"] == doc_id, "text"].iloc[0]
    print(f"[{score:.3f}] {doc_id}: {highlight_snippet(text, prep_en('hubble telescope mirror'))}")

The [Shuttle] [launch]ed today


[0.392] en-01360: …s, but have you looked ad the [Hubble] results, the defects of the [mirror] are partially corr…
[0.344] en-00411: …all >1g forces applied to the [mirror] and its mounting (and nearly all =1g forces) have bee…


`(?<![\w\[])`는 앞 글자가 단어 글자도 아니고 `[`도 아닐 때만 찾으라는 뜻입니다. 이미 감싼 단어를 다시 감싸지 않게 합니다.

## ✍️ 실습 2: 나만의 품질 점검

영어와 한국어에서 쿼리를 각각 3개 이상 직접 만들고, 기대 카테고리와 함께 `precision_at_k`로 점검하세요. 그중 precision@5가 가장 낮은 쿼리 하나를 골라 `explain`으로 1위 문서의 기여도를 확인하고, 실패 원인을 16.8절 표의 유형 중 하나로 분류해 보세요.

In [20]:
# 여기에 작성하세요

**정답 예시**

In [21]:
my_checks = [
    (engine_en, labels_en, "orbit satellite moon", "sci.space"),
    (engine_en, labels_en, "image format conversion", "comp.graphics"),
    (engine_en, labels_en, "season record team", "rec.sport.baseball"),
    (engine_ko, labels_ko, "금리 인상 은행", "경제"),
    (engine_ko, labels_ko, "야구 홈런 투수", "스포츠"),
    (engine_ko, labels_ko, "선거 후보 공천", "정치"),
]
rows = [(q, exp, precision_at_k(eng, lab, q, exp)[0]) for eng, lab, q, exp in my_checks]
for q, exp, p in rows:
    print(f"{q:<26} 기대 {exp:<20} precision@5 {p:.1f}")

worst = min(rows, key=lambda r: r[2])
eng = engine_en if worst[0].isascii() else engine_ko
print(f"\n가장 낮은 쿼리: {worst[0]!r}\n")
explain(eng, worst[0])

orbit satellite moon       기대 sci.space            precision@5 1.0
image format conversion    기대 comp.graphics        precision@5 1.0
season record team         기대 rec.sport.baseball   precision@5 0.0
금리 인상 은행                   기대 경제                   precision@5 0.8
야구 홈런 투수                   기대 스포츠                  precision@5 1.0
선거 후보 공천                   기대 정치                   precision@5 0.8

가장 낮은 쿼리: 'season record team'

=== 코사인 유사도 계산 예시 ===
Query: "season record team"
전처리 후 토큰: ['season', 'record', 'team']
대상 문서: 1위 en-01448 [rec.sport.hockey] "… the score for each team equal. Each team gets one point for…"
벡터 차원: 13,387 (쿼리에서 0이 아닌 차원 3개, 문서에서 62개)

단어               IDF     q 가중치     d 가중치       q×d
record         4.819    0.6944    0.1884    0.1308
season         3.791    0.5464    0.0000    0.0000
team           3.250    0.4683    0.3812    0.1785

코사인 유사도 = Σ q×d = 0.3093  (정규화된 벡터라 분모는 1 × 1)


가장 낮은 쿼리의 기여도 표를 보면, 기대와 다른 카테고리 문서가 **어느 단어 덕분에** 올라왔는지 알 수 있습니다. 여러 주제에서 함께 쓰이는 단어(예: `season`, `record`, `team`은 야구와 하키 모두)가 원인이라면, 이것은 단어 하나하나만 보고 조합의 의미를 보지 못하는 BoW의 한계입니다.

## 🧐 퀴즈

**Q1.** 쿼리가 들어올 때마다 쿼리를 포함해 TF-IDF를 다시 `fit`하면 어떤 문제가 생기나요?
<details><summary>▶ 정답 보기</summary>

어휘와 IDF가 쿼리마다 바뀌어 저장해 둔 문서 행렬과 열(단어)·가중치가 맞지 않게 됩니다. 문서 행렬도 매번 다시 계산해야 해서 느립니다. 문서로 한 번 `fit`하고 쿼리는 `transform`만 해야 합니다.
</details>

**Q2.** 한국어 문서를 Okt로 색인했는데 쿼리를 공백으로 자르면 왜 결과가 거의 나오지 않나요?
<details><summary>▶ 정답 보기</summary>

어휘 사전에는 `삼성`, `실적` 같은 형태소가 들어 있는데, 공백 토큰은 `삼성전자가`, `실적을`처럼 조사가 붙은 어절이라 글자가 달라 어휘에서 찾을 수 없기 때문입니다.
</details>

**Q3.** 검색 결과에서 점수가 0인 문서를 빼는 이유는?
<details><summary>▶ 정답 보기</summary>

점수 0은 쿼리 단어가 하나도 없다는 뜻이라 관련 없는 문서입니다. 상위 k개를 채우려고 넣으면 관련 있는 것처럼 보여 사용자를 오해하게 만듭니다.
</details>

**Q4.** `"북한"`과 `"北"`의 검색 결과가 거의 겹치지 않는 이유와 개선 방법은?
<details><summary>▶ 정답 보기</summary>

BoW에서 서로 다른 단어는 서로 다른 차원이라, 뜻이 같아도 겹치는 부분이 없습니다. 동의어 사전으로 쿼리를 확장하거나(북한 → 북한 + 北), 단어 임베딩처럼 의미가 비슷한 단어를 가까운 벡터로 표현하는 방법을 쓸 수 있습니다.
</details>

**Q5.** 스니펫을 만들 때 `text.find(단어)` 대신 `(?<!\w)단어` 정규식을 쓰는 이유는?
<details><summary>▶ 정답 보기</summary>

`find`는 다른 단어 안의 일부(`start` 안의 `art`)도 찾기 때문입니다. `(?<!\w)`는 단어가 시작하는 위치에서만 찾게 합니다. 끝은 막지 않아서 `launch`로 `launched`도 찾을 수 있습니다.
</details>

## ✅ 체크포인트
- [ ] `search(query, topk)`가 `(점수, 문서 ID, 스니펫)` 리스트를 돌려준다.
- [ ] 쿼리에 문서와 같은 전처리를 적용하고, 그 이유를 실패 예로 설명할 수 있다.
- [ ] 잘못된 `topk`, 점수 0 문서, 0 벡터 쿼리를 처리했다.
- [ ] `argparse` CLI를 만들고 README에 실행 방법(안 A, 안 B)을 쓸 수 있다.
- [ ] 품질 점검표와 실패 쿼리 표를 만들고 BoW 한계와 연결할 수 있다.
- [ ] 1위 문서의 점수를 단어별 기여도로 분해해 설명할 수 있다.

## 🔗 과제 연결과 다음 장
- `SearchEngine`, `search`, `top_k`, `make_snippet`, `format_results`는 `src/search.py`가 되고, CLI는 `main.py --mode search`가 됩니다. 과제 요구사항 "문서 검색 시스템"과 "실행 방법 명시"를 이 장에서 완성했습니다.
- 16.8절 실패 쿼리는 21장 BoW 한계 분석의 사례로 다시 씁니다.
- **17장. 머신러닝 기초**부터는 같은 TF-IDF 벡터로 문서의 주제를 맞히는 분류 실험을 합니다.